In [ ]:
# @title 🚀 AI 单图转 Live2D 三大路线一键全跑工作台（Single-Cell 全修复满血版）
# @markdown **点一次 `▶` 一镜到底跑完**（支持 Colab 免费 `T4 16GB` / `L4` / `A100` / `G4 96GB Blackwell`）：
# @markdown - **路线 1（真 Live2D 工程）**：`shitagaki-lab/see-through` (SIGGRAPH 2026) + 双眼倾角对齐 5 层纯英文标准口腔/牙/圆润 U 形舌合成器 + `tsunehimatoi/psd2live` 导出 `.psd` / `.cmo3` / `.moc3` / `VTube Studio` / `WebGL` / 物理动画
# @markdown - **路线 2（45 参数神经人偶）**：`Talking Head Anime 4`（自动无损前景抠图 + `face.png` 精准红框对齐，Dropbox 限流自动切 `THA3`）
# @markdown - **路线 3（隐式关键点重演）**：`KlingAIResearch/LivePortrait`（`d0`/`d11` 驱动重演 + 零视频类 Live2D 滑块扫参）

RUN_ROUTE_1 = True  # @param {type:"boolean"}
RUN_ROUTE_2 = True  # @param {type:"boolean"}
RUN_ROUTE_3 = True  # @param {type:"boolean"}
USE_CUSTOM_IMAGE = False  # @param {type:"boolean"}
ROUTE1_IMAGE_MODE = "bust"  # @param ["bust", "full"]
AUTO_SYNTH_MOUTH = True  # @param {type:"boolean"}
ROUTE1_STEPS = 30  # @param {type:"integer"}
REUSE_CACHE_IF_EXISTS = True  # @param {type:"boolean"}
SAVE_TO_DRIVE = False  # @param {type:"boolean"}

import base64
import hashlib
import json
import os
import shutil
import subprocess
import sys
import traceback
import urllib.request
import zipfile
from pathlib import Path

os.environ["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

import torch
from IPython.display import HTML, display

if not torch.cuda.is_available():
    raise RuntimeError("未检测到 NVIDIA CUDA GPU！请在 Colab 顶部菜单选择【代码执行程序 -> 更改运行时类型 -> GPU】。")

gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / (1024 ** 3)
native_bf16 = props.major >= 8
use_bf16_full = (vram_gb >= 20.0 and native_bf16)

WORK = Path("/content/ai_live2d_3routes")
OUT_ALL = WORK / "outputs"
ASSETS_DIR = WORK / "assets"
SCRIPTS_DIR = WORK / "_scripts"
for _d in (OUT_ALL, ASSETS_DIR, SCRIPTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print("=" * 74)
print(f"🚀 GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | sm_{props.major}{props.minor} | PyTorch {torch.__version__}")
print(f"⚙️  路线 1 档位: {'原生满血 BF16 + 1280px (零量化)' if use_bf16_full else 'NF4 4-bit 量化 + 1024px (T4 安全模式)'}")
print("=" * 74)

# 0. 前置交互操作（避免跑完依赖才弹窗导致无人值守卡住）
DRIVE_OUT_DIR = None
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        DRIVE_OUT_DIR = Path("/content/drive/MyDrive/AI_Live2D_Outputs")
        DRIVE_OUT_DIR.mkdir(parents=True, exist_ok=True)
        print(f"☁️ Google Drive 已挂载，成品将同步备份至: {DRIVE_OUT_DIR}")
    except Exception as _e:
        print(f"⚠️ Google Drive 挂载跳过: {_e}")

UPLOADED_RAW_PATH = None
if USE_CUSTOM_IMAGE:
    from google.colab import files as colab_files
    print("📤 请上传一张二次元正面角色单图 (PNG / JPG / WEBP)：")
    _up = colab_files.upload()
    if not _up:
        raise RuntimeError("未选择任何上传图片！")
    _fname = next(iter(_up.keys()))
    UPLOADED_RAW_PATH = ASSETS_DIR / _fname
    UPLOADED_RAW_PATH.write_bytes(_up[_fname])

def run_stream(cmd, cwd=None, check=True, env=None):
    show = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    print(f"\n>>> $ {show}", flush=True)
    proc = subprocess.Popen(
        cmd, shell=isinstance(cmd, str), cwd=cwd, env=env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if check and proc.returncode != 0:
        raise RuntimeError(f"命令执行失败 (exit={proc.returncode}): {show}")
    return proc.returncode

def show_img_b64(img_path: Path, title: str, max_w: int = 920):
    if not img_path.exists():
        return
    ext = img_path.suffix.lstrip(".").lower()
    mime = "image/gif" if ext == "gif" else ("image/jpeg" if ext in ("jpg", "jpeg") else "image/png")
    b64 = base64.b64encode(img_path.read_bytes()).decode("ascii")
    display(HTML(
        f'<div style="margin:10px 0;"><div style="font-weight:700;margin-bottom:4px;">🖼️ {title}</div>'
        f'<img src="data:{mime};base64,{b64}" style="max-width:{max_w}px;border-radius:8px;border:1px solid #ddd;"/></div>'
    ))

def show_video_b64(video_path: Path, title: str, width: int = 480):
    if not video_path.exists():
        return
    play_p = video_path
    sz_mb = video_path.stat().st_size / (1024 * 1024)
    if sz_mb > 9.0:
        tmp_p = video_path.with_name(video_path.stem + "_web.mp4")
        subprocess.run([
            "ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path),
            "-vf", "scale='min(720,iw)':-2", "-c:v", "libx264", "-crf", "24",
            "-pix_fmt", "yuv420p", "-c:a", "aac", "-b:a", "96k", str(tmp_p),
        ], check=False)
        if tmp_p.exists() and tmp_p.stat().st_size > 0:
            play_p = tmp_p
    b64 = base64.b64encode(play_p.read_bytes()).decode("ascii")
    display(HTML(
        f'<div style="display:inline-block;margin:8px;vertical-align:top;">'
        f'<div style="font-weight:700;margin-bottom:4px;">🎬 {title} ({sz_mb:.1f} MB)</div>'
        f'<video width="{width}" controls autoplay loop muted playsinline '
        f'style="border-radius:8px;border:1px solid #ddd;background:#111;">'
        f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video></div>'
    ))

# 1. 一次性安装兼容依赖（不碰预装 PyTorch；强制使用纯净 CPU 版 onnxruntime 杜绝 Blackwell/Py3.13 段错误）
print("\n=== [Step 1] 安装系统工具与 Python 依赖（免重启单格闭环）===")
run_stream("apt-get update -qq && apt-get install -y -qq ffmpeg libgl1 libegl1 libglib2.0-0 fonts-wqy-zenhei")

pkgs = [
    "imageio>=2.34.0", "imageio-ffmpeg>=0.5.1", "ffmpeg-python==0.2.0",
    "huggingface-hub==1.7.2", "safetensors==0.7.0", "omegaconf==2.3.0",
    "einops==0.8.2", "rich>=13.7.1", "opencv-python-headless", "matplotlib",
]
if RUN_ROUTE_1:
    pkgs += [
        "diffusers==0.37.0", "transformers==5.0.0", "accelerate==1.13.0",
        "psd-tools[composite]==1.14.2", "kornia==0.8.2", "pycocotools", "scikit-learn", "scipy",
    ]
    if not use_bf16_full:
        pkgs.append("bitsandbytes==0.49.2")
if RUN_ROUTE_2 and not RUN_ROUTE_1:
    pkgs.append("rembg")
if RUN_ROUTE_3:
    pkgs += ["onnx>=1.16.1", "tyro==0.8.5", "pykalman>=0.10.1"]

run_stream([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", *pkgs])
if RUN_ROUTE_2 or RUN_ROUTE_3:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "onnxruntime-gpu"], check=False)
    run_stream([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "onnxruntime"])

from PIL import Image

# 2. 准备输入立绘
print("\n=== [Step 2] 准备统一测试立绘 ===")
KIMI_URL = (
    "https://raw.githubusercontent.com/lmy414/ai-chan-sheets/main/"
    "%E5%8F%82%E8%80%83%E5%8E%9F%E5%9B%BE/06_Kimi%E5%A8%98_Moonshot/"
    "%E5%8E%9F%E5%9B%BE_%E7%AB%8B%E7%BB%98_%E5%85%A8%E8%BA%AB.png"
)

def to_rgb_white(im: Image.Image) -> Image.Image:
    if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
        rgba = im.convert("RGBA")
        bg = Image.new("RGB", rgba.size, (255, 255, 255))
        bg.paste(rgba, mask=rgba.split()[3])
        return bg
    return im.convert("RGB")

def make_bust_1024(im: Image.Image) -> Image.Image:
    w, h = im.size
    side = min(w, int(h * 0.52)) if h > int(w * 1.25) else min(w, h)
    x0 = max(0, (w - side) // 2)
    return im.crop((x0, 0, x0 + side, side)).resize((1024, 1024), Image.Resampling.LANCZOS)

SRC_FULL = ASSETS_DIR / "source_full.png"
SRC_BUST = ASSETS_DIR / "source_bust_1024.png"
if UPLOADED_RAW_PATH is not None:
    raw_im = Image.open(UPLOADED_RAW_PATH)
    if max(raw_im.size) > 2048:
        sc = 2048.0 / max(raw_im.size)
        raw_im = raw_im.resize((int(raw_im.width * sc), int(raw_im.height * sc)), Image.Resampling.LANCZOS)
    raw_im.save(SRC_FULL)
    to_rgb_white(make_bust_1024(raw_im)).save(SRC_BUST)
    MODEL_NAME = "".join(ch if ch.isalnum() or ch in ("_", "-") else "_" for ch in UPLOADED_RAW_PATH.stem).strip("_") or "custom_char"
else:
    if not SRC_FULL.exists():
        req = urllib.request.Request(KIMI_URL, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=30) as resp:
            SRC_FULL.write_bytes(resp.read())
    full_im = Image.open(SRC_FULL)
    to_rgb_white(make_bust_1024(full_im)).save(SRC_BUST)
    MODEL_NAME = "kimi_chan"

st_input_img = SRC_BUST if ROUTE1_IMAGE_MODE == "bust" else SRC_FULL
img_sha1 = hashlib.sha1(st_input_img.read_bytes()).hexdigest()[:12]
print(f"✅ 素材就绪: {MODEL_NAME} | full={Image.open(SRC_FULL).size} | bust={Image.open(SRC_BUST).size} | sha1={img_sha1}")
show_img_b64(st_input_img, f"基准输入立绘 ({MODEL_NAME}, mode={ROUTE1_IMAGE_MODE})", max_w=360)

STATUS = {}

# ==============================================================================
# 路线 1：See-through + 纯英文标准 5 层口腔/上牙/圆润 U 形舌 + PSD2Live 真 Live2D
# ==============================================================================
if RUN_ROUTE_1:
    print("\n" + "#" * 74)
    print("### [路线 1] See-through + 5层口腔/牙/U形舌 + PSD2Live (.psd / .cmo3 / .moc3)")
    print("#" * 74)
    try:
        R1_DIR = OUT_ALL / "01_seethrough_psd2live"
        ST_OUT = R1_DIR / "seethrough_layers"
        R1_DIR.mkdir(parents=True, exist_ok=True)

        ST_REPO = Path("/content/see-through")
        if not ST_REPO.exists():
            run_stream(f"git clone --depth 1 https://github.com/shitagaki-lab/see-through.git {ST_REPO}")
        if not (ST_REPO / "assets").exists():
            os.symlink(ST_REPO / "common" / "assets", ST_REPO / "assets")
        run_stream([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(ST_REPO / "common"), "-e", str(ST_REPO / "annotators")])

        P2L_ROOT = Path("/content/psd2live_app")
        P2L_BIN = P2L_ROOT / "PSD2Live" / "bin" / "PSD2Live"
        if not P2L_BIN.exists():
            P2L_ROOT.mkdir(parents=True, exist_ok=True)
            url_p2l = "https://github.com/tsunehimatoi/psd2live/releases/download/v3.8.4/PSD2Live-3.8.4-linux-x64.tar.gz"
            tar_p = P2L_ROOT / "psd2live.tar.gz"
            run_stream(f'curl -L --fail --retry 3 -o "{tar_p}" "{url_p2l}"')
            run_stream(f'tar -xzf "{tar_p}" -C "{P2L_ROOT}"')
            tar_p.unlink(missing_ok=True)
            os.chmod(P2L_BIN, 0o755)

        cache_ok = False
        meta_file = ST_OUT / "canvas_meta.json"
        if REUSE_CACHE_IF_EXISTS and (ST_OUT / "face.png").exists() and meta_file.exists():
            try:
                old_meta = json.loads(meta_file.read_text(encoding="utf-8"))
                old_sha = old_meta.get("input_sha1")
                if old_sha in (None, img_sha1):
                    cache_ok = True
            except Exception:
                cache_ok = False

        if cache_ok:
            print(f"⚡ 检测到已计算的 See-through 23 层缓存 ({ST_OUT})，直接秒级复用跳过扩散！")
        else:
            ST_SCRIPT = SCRIPTS_DIR / "run_seethrough_v2.py"
            ST_SCRIPT.write_text(r"""
import argparse, gc, os, os.path as osp
import cv2, numpy as np, torch
from PIL import Image
from modules.layerdiffuse.diffusers_kdiffusion_sdxl import KDiffusionStableDiffusionXLPipeline
from modules.layerdiffuse.layerdiff3d import UNetFrameConditionModel
from modules.layerdiffuse.vae import TransparentVAE
from modules.marigold import MarigoldDepthPipeline
from utils.cv import center_square_pad_resize, smart_resize, validate_resolution
from utils.io_utils import dict2json
from utils.torch_utils import seed_everything

VALID_V2 = ["hair","headwear","face","eyes","eyewear","ears","earwear","nose","mouth","neck","neckwear","topwear","handwear","bottomwear","legwear","footwear","tail","wings","objects"]
BODY_TAGS = ["front hair","back hair","head","neck","neckwear","topwear","handwear","bottomwear","legwear","footwear","tail","wings","objects"]
HEAD_TAGS = ["headwear","face","irides","eyebrow","eyewhite","eyelash","eyewear","ears","earwear","nose","mouth"]
ALL_V3 = ["front hair","back hair","headwear","face","irides","eyebrow","eyewhite","eyelash","eyewear","ears","earwear","nose","mouth","neck","neckwear","topwear","handwear","bottomwear","legwear","footwear","tail","wings","objects"]
REPOS = {
    "none": {"ld": "layerdifforg/seethroughv0.0.2_layerdiff3d", "dp": "24yearsold/seethroughv0.0.1_marigold"},
    "nf4":  {"ld": "24yearsold/seethroughv0.0.2_layerdiff3d_nf4", "dp": "24yearsold/seethroughv0.0.1_marigold_nf4"},
}

def alpha_over(layers):
    h, w = layers[0].shape[:2]
    rgb = np.zeros((h, w, 3), np.float32)
    a = np.zeros((h, w, 1), np.float32)
    for lyr in layers:
        fr, fa = lyr[..., :3].astype(np.float32) / 255.0, lyr[..., 3:4].astype(np.float32) / 255.0
        inv = 1.0 - fa
        na = fa + a * inv
        rgb = np.divide(fr * fa + rgb * a * inv, np.maximum(na, 1e-6))
        a = na
    return np.clip(np.round(np.concatenate([rgb, a], axis=-1) * 255.0), 0, 255).astype(np.uint8)

def is_quant(m):
    return m is not None and any([getattr(m, k, False) for k in ("is_quantized","is_loaded_in_4bit","is_loaded_in_8bit")] + [getattr(m, "quantization_method", None) is not None])

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--srcp", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--quant", default="none", choices=["none", "nf4"])
    ap.add_argument("--res", type=int, default=1280)
    ap.add_argument("--steps", type=int, default=30)
    ap.add_argument("--sha1", default="")
    args = ap.parse_args()
    os.makedirs(args.out, exist_ok=True)
    seed_everything(42)

    repo_ld = REPOS[args.quant]["ld"]
    print(f"[See-through] 加载 LayerDiff3D: {repo_ld} (res={args.res}, steps={args.steps})", flush=True)
    unet = UNetFrameConditionModel.from_pretrained(repo_ld, subfolder="unet")
    tvae = TransparentVAE.from_pretrained(repo_ld, subfolder="trans_vae")
    pipe = KDiffusionStableDiffusionXLPipeline.from_pretrained(repo_ld, trans_vae=tvae, unet=unet, scheduler=None)
    for m in (pipe.vae, pipe.trans_vae, pipe.unet):
        if not is_quant(m):
            m.to(device="cuda", dtype=torch.bfloat16)
    for ename in ("text_encoder", "text_encoder_2"):
        enc = getattr(pipe, ename, None)
        if enc is not None and hasattr(enc, "dequantize") and is_quant(enc):
            setattr(pipe, ename, enc.dequantize().to(device="cuda", dtype=torch.bfloat16))
        elif enc is not None and not is_quant(enc):
            enc.to(device="cuda", dtype=torch.bfloat16)
    pipe.cache_tag_embeds(unload_textencoders=True)
    gc.collect(); torch.cuda.empty_cache()

    inp = np.array(Image.open(args.srcp).convert("RGBA"))
    src_h, src_w = inp.shape[:2]
    fullpage, bpad_sz, bpad_pos = center_square_pad_resize(inp, args.res, return_pad_info=True)
    bscale = bpad_sz[0] / float(args.res)
    Image.fromarray(fullpage).save(osp.join(args.out, "src_img.png"))
    rng = torch.Generator(device="cuda").manual_seed(42)

    out_b = pipe(strength=1.0, num_inference_steps=args.steps, batch_size=1, generator=rng,
                 guidance_scale=1.0, prompt=BODY_TAGS, negative_prompt="", fullpage=fullpage, group_index=0)
    for im, tag in zip(out_b.images, BODY_TAGS):
        Image.fromarray(im).save(osp.join(args.out, f"{tag}.png"))

    hmask = (out_b.images[2][..., -1] > 15).astype(np.uint8)
    nz = cv2.findNonZero(hmask)
    if nz is None:
        hx1, hy1, hx2, hy2 = 0, 0, src_w, max(1, int(src_h * 0.6))
    else:
        hx0, hy0, hw0, hh0 = cv2.boundingRect(nz)
        hx = max(0, min(int(round(hx0 * bscale)) - bpad_pos[0], src_w - 1))
        hy = max(0, min(int(round(hy0 * bscale)) - bpad_pos[1], src_h - 1))
        hxe, hye = min(src_w, hx + max(1, int(round(hw0 * bscale)))), min(src_h, hy + max(1, int(round(hh0 * bscale))))
        we, he = max(1, hxe - hx), max(1, hye - hy)
        hx1, hx2 = max(0, hx - int(we * 0.30)), min(src_w, hxe + int(we * 0.30))
        hy1, hy2 = max(0, hy - int(he * 0.60)), min(src_h, hye + int(he * 0.30))

    in_head = inp[hy1:hy2, hx1:hx2]
    ih, iw = in_head.shape[:2]
    hsq, hpad_sz, hpad_pos = center_square_pad_resize(in_head, args.res, return_pad_info=True)
    out_h = pipe(strength=1.0, num_inference_steps=args.steps, batch_size=1, generator=rng,
                 guidance_scale=1.0, prompt=HEAD_TAGS, negative_prompt="", fullpage=hsq, group_index=1)

    cx1 = max(0, min(args.res - 1, int(round((hx1 + bpad_pos[0]) / bscale))))
    cy1 = max(0, min(args.res - 1, int(round((hy1 + bpad_pos[1]) / bscale))))
    cx2 = max(cx1 + 1, min(args.res, int(round((hx2 + bpad_pos[0]) / bscale))))
    cy2 = max(cy1 + 1, min(args.res, int(round((hy2 + bpad_pos[1]) / bscale))))
    hside = float(hpad_sz[0])
    rx1 = max(0, int(round(hpad_pos[0] * args.res / hside)))
    ry1 = max(0, int(round(hpad_pos[1] * args.res / hside)))
    rx2 = min(args.res, max(rx1 + 1, int(round((hpad_pos[0] + iw) * args.res / hside))))
    ry2 = min(args.res, max(ry1 + 1, int(round((hpad_pos[1] + ih) * args.res / hside))))
    for rst, tag in zip(out_h.images, HEAD_TAGS):
        rs = smart_resize(rst[ry1:ry2, rx1:rx2], (cy2 - cy1, cx2 - cx1))
        canvas = np.zeros((args.res, args.res, 4), np.uint8)
        canvas[cy1:cy1 + rs.shape[0], cx1:cx1 + rs.shape[1]] = rs
        Image.fromarray(canvas).save(osp.join(args.out, f"{tag}.png"))

    vx1 = int(round(bpad_pos[0] / bscale)); vy1 = int(round(bpad_pos[1] / bscale))
    vx2 = min(args.res, int(round((bpad_pos[0] + src_w) / bscale)))
    vy2 = min(args.res, int(round((bpad_pos[1] + src_h) / bscale)))
    dict2json({"valid_content_xyxy": [vx1, vy1, vx2, vy2], "input_sha1": args.sha1}, osp.join(args.out, "canvas_meta.json"))

    del pipe, unet, tvae; gc.collect(); torch.cuda.empty_cache()

    repo_dp = REPOS[args.quant]["dp"]
    print(f"[See-through] 加载 Marigold Depth: {repo_dp}", flush=True)
    if args.quant == "none":
        mu = UNetFrameConditionModel.from_pretrained(repo_dp, subfolder="unet")
        mp = MarigoldDepthPipeline.from_pretrained(repo_dp, unet=mu).to(device="cuda", dtype=torch.bfloat16)
    else:
        mu = UNetFrameConditionModel.from_pretrained(repo_dp, subfolder="unet", torch_dtype=torch.bfloat16)
        mp = MarigoldDepthPipeline.from_pretrained(repo_dp, unet=mu, torch_dtype=torch.bfloat16)
        for m in (mp.vae, mp.unet):
            if not is_quant(m):
                m.to(device="cuda", dtype=torch.bfloat16)
        if hasattr(mp.text_encoder, "dequantize") and is_quant(mp.text_encoder):
            mp.text_encoder = mp.text_encoder.dequantize().to(device="cuda", dtype=torch.bfloat16)
    mp.cache_tag_embeds(unload_textencoders=True)

    fp = np.array(Image.open(osp.join(args.out, "src_img.png")).convert("RGBA"))
    sh, sw = fp.shape[:2]
    dres = validate_resolution(768)
    empty = np.zeros((sh, sw, 4), np.uint8)
    imgs = []
    for t in VALID_V2:
        p = osp.join(args.out, f"{t}.png")
        if osp.exists(p):
            a = np.array(Image.open(p).convert("RGBA")); a[..., -1][a[..., -1] < 15] = 0; imgs.append(a)
        else:
            imgs.append(empty)
    cdict = {}
    for c, cl in {"eyes": ["eyewhite","irides","eyelash","eyebrow"], "hair": ["back hair","front hair"]}.items():
        il, tl = [], []
        for t in cl:
            p = osp.join(args.out, f"{t}.png")
            if osp.exists(p):
                a = np.array(Image.open(p).convert("RGBA")); a[..., -1][a[..., -1] < 15] = 0; il.append(a); tl.append(t)
        if il:
            imgs[VALID_V2.index(c)] = alpha_over(il); cdict[c] = (tl, il)
    ba = np.zeros((sh, sw), np.float32)
    for im in imgs:
        ba += im[..., -1].astype(np.float32) / 255.0
    fp[..., -1] = (np.clip(ba, 0, 1) * 255).astype(np.uint8); imgs.append(fp)
    d_in = [smart_resize(im, dres) for im in imgs]
    seed_everything(42)
    dp_pred = [smart_resize(d, (sh, sw)) for d in mp(color_map=None, img_list=d_in).depth_tensor.cpu().float().numpy()]
    for ii, d in enumerate(dp_pred[:-1]):
        du8 = (np.clip(d, 0, 1) * 255).astype(np.uint8)
        t = VALID_V2[ii]
        if t in cdict:
            occ = np.zeros((sh, sw), bool)
            for st, im in zip(cdict[t][0][::-1], cdict[t][1][::-1]):
                ml = im[..., -1] > 15; mi = occ & ml; mv = ml & (~mi)
                dl = np.full((sh, sw), 255, np.uint8); dl[ml] = du8[ml]
                if np.any(mi):
                    fd = int(np.round(np.median(du8[mv]))) if np.any(mv) else (int(np.round(np.median(du8[ml]))) if np.any(ml) else 255)
                    dl[mi] = np.clip(fd, 0, 255)
                occ |= ml
                Image.fromarray(dl).save(osp.join(args.out, f"{st}_depth.png"))
        else:
            Image.fromarray(du8).save(osp.join(args.out, f"{t}_depth.png"))

    recs = []
    for t in ALL_V3:
        p, dp = osp.join(args.out, f"{t}.png"), osp.join(args.out, f"{t}_depth.png")
        if osp.exists(p) and osp.exists(dp):
            arr = np.array(Image.open(p).convert("RGBA")); m = arr[..., -1] > 15
            if int(m.sum()) >= 16:
                dm = float(np.median(np.array(Image.open(dp).convert("L"), np.float32)[m] / 255.0))
                recs.append({"tag": t, "img": arr, "dm": dm})
    recs.sort(key=lambda x: x["dm"], reverse=True)
    if recs:
        Image.fromarray(alpha_over([r["img"] for r in recs])).save(osp.join(args.out, "reconstruction.png"))
    print(f"[See-through] 完成！有效非空图层: {len(recs)} / 23", flush=True)

if __name__ == "__main__":
    main()
""".lstrip(), encoding="utf-8")

            run_stream([
                sys.executable, "-u", str(ST_SCRIPT),
                "--srcp", str(st_input_img), "--out", str(ST_OUT),
                "--quant", "none" if use_bf16_full else "nf4",
                "--res", "1280" if use_bf16_full else "1024",
                "--steps", str(ROUTE1_STEPS), "--sha1", img_sha1,
            ], cwd=str(ST_REPO))

        # 独立子进程执行 PSD 去噪组装（纯 ASCII 层名防 mac_roman 截断 + 圆润 U 形舌）与 PSD2Live 导出
        P2L_SCRIPT = SCRIPTS_DIR / "run_psd_and_psd2live.py"
        P2L_SCRIPT.write_text(r"""
import argparse, json, math, shutil, subprocess
from pathlib import Path
import cv2
import numpy as np
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from psd_tools import PSDImage

ap = argparse.ArgumentParser()
ap.add_argument("--r1-dir", required=True)
ap.add_argument("--p2l-bin", required=True)
ap.add_argument("--model-name", required=True)
ap.add_argument("--mode", default="bust")
ap.add_argument("--auto-mouth", default="1")
args = ap.parse_args()

R1_DIR = Path(args.r1_dir)
ST_OUT = R1_DIR / "seethrough_layers"
P2L_BIN = Path(args.p2l_bin)
MODEL_NAME = args.model_name
AUTO_MOUTH = (args.auto_mouth == "1")

meta = json.loads((ST_OUT / "canvas_meta.json").read_text(encoding="utf-8"))
vx1, vy1, vx2, vy2 = meta["valid_content_xyxy"]
cw, ch = vx2 - vx1, vy2 - vy1

# handwear 置于 topwear 之下（袖子收纳进披肩/外套下），严守 Live2D 解剖层级
ANATOMY = [
    "back hair", "wings", "tail", "footwear", "legwear", "bottomwear",
    "handwear", "topwear", "neck", "neckwear", "ears", "earwear",
    "face", "nose", "mouth", "eyewhite", "irides", "eyelash", "eyebrow",
    "eyewear", "front hair", "headwear", "objects",
]
BILATERAL = {"eyewhite", "irides", "eyelash", "eyebrow", "ears", "handwear"}
if args.mode == "full":
    BILATERAL.update({"legwear", "footwear"})

def clean_layer(arr, alpha_th=55, min_comp=18):
    out = arr.copy()
    m = (out[..., 3] >= alpha_th).astype(np.uint8)
    nl, lbl, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    keep = np.zeros_like(m, dtype=bool)
    for k in range(1, nl):
        if st[k, cv2.CC_STAT_AREA] >= min_comp:
            keep |= (lbl == k)
    out[~keep, 3] = 0
    return out

raw_lyrs = {}
for t in ANATOMY:
    p = ST_OUT / f"{t}.png"
    if not p.exists():
        continue
    arr = np.array(Image.open(p).convert("RGBA"))[vy1:vy2, vx1:vx2]
    arr = clean_layer(arr, alpha_th=55, min_comp=18)
    if int((arr[..., 3] >= 55).sum()) < 45:
        continue
    if t == "nose":
        rgb_m = arr[arr[..., 3] >= 120][:, :3]
        if len(rgb_m) > 0 and float(rgb_m.mean()) > 228.0:
            print("ℹ️ 跳过 nose 层（滤除白色皮肤斑块伪影，保留 face 原生鼻影）")
            continue
    raw_lyrs[t] = arr

axis_x = float(np.median(np.where(raw_lyrs["face"][..., 3] >= 60)[1])) if "face" in raw_lyrs else cw / 2.0

def split_lr(rgba, sx):
    m = (rgba[..., 3] >= 55).astype(np.uint8)
    tot = int(m.sum())
    if tot < 32:
        return None
    nl, lbl, st, cen = cv2.connectedComponentsWithStats(m, connectivity=8)
    mr, ml, ix = np.zeros_like(m, bool), np.zeros_like(m, bool), int(round(sx))
    for k in range(1, nl):
        c = lbl == k
        x0, w0 = st[k, cv2.CC_STAT_LEFT], st[k, cv2.CC_STAT_WIDTH]
        if x0 < ix < x0 + w0 and w0 > 12:
            mr[:, :ix] |= c[:, :ix]; ml[:, ix:] |= c[:, ix:]
        elif float(cen[k][0]) < sx:
            mr |= c
        else:
            ml |= c
    if int(mr.sum()) < max(16, tot * 0.05) or int(ml.sum()) < max(16, tot * 0.05):
        return None
    r_out, l_out = rgba.copy(), rgba.copy()
    r_out[~mr] = 0; l_out[~ml] = 0
    return r_out, l_out

def trim_rgba(rgba, pad=2):
    ys, xs = np.where(rgba[..., 3] > 0)
    if len(xs) == 0:
        return None
    h, w = rgba.shape[:2]
    x1, y1 = max(0, int(xs.min()) - pad), max(0, int(ys.min()) - pad)
    x2, y2 = min(w, int(xs.max()) + 1 + pad), min(h, int(ys.max()) + 1 + pad)
    return rgba[y1:y2, x1:x2].copy(), x1, y1

def synth_anime_mouth_ascii(m_rgba, f_rgba, lyrs, sx):
    m = m_rgba[..., 3] >= 60
    ys, xs = np.where(m)
    if len(xs) < 10:
        return None
    mw = max(1.0, float(xs.max() - xs.min() + 1))
    ml_x = int(xs.min()) + int(mw * 0.25)
    mr_x = max(ml_x + 1, int(xs.max()) - int(mw * 0.25))
    spans = [float(np.where(m[:, cx_i])[0].max() - np.where(m[:, cx_i])[0].min() + 1) for cx_i in range(ml_x, mr_x + 1) if np.any(m[:, cx_i])]
    if spans and (float(np.median(spans)) / mw) >= 0.45:
        return None

    roll = 0.0
    for et in ("irides", "eyewhite", "eyelash"):
        if et in lyrs:
            ey, ex = np.where(lyrs[et][..., 3] >= 60)
            ls, rs = ex < sx, ex >= sx
            if int(ls.sum()) >= 15 and int(rs.sum()) >= 15:
                lx, ly = float(np.median(ex[ls])), float(np.median(ey[ls]))
                rx, ry = float(np.median(ex[rs])), float(np.median(ey[rs]))
                if rx - lx > 20:
                    roll = float(np.clip(math.atan2(ry - ly, rx - lx), -math.radians(20), math.radians(20)))
                    break

    H, W = m_rgba.shape[:2]
    px = m_rgba[m][:, :3].astype(np.float32)
    luma = 0.299 * px[:, 0] + 0.587 * px[:, 1] + 0.114 * px[:, 2]
    s_rgb = np.median(px[np.argsort(luma)[:max(5, len(luma) // 4)]], axis=0)
    lrgb = np.clip(s_rgb * 0.72 + np.array([82.0, 44.0, 50.0]) * 0.28, 28, 138).astype(np.uint8)
    lcol = (int(lrgb[0]), int(lrgb[1]), int(lrgb[2]), 255)

    cx, cy = float(np.median(xs)), float(np.median(ys))
    if f_rgba is not None and np.any(f_rgba[..., 3] >= 60):
        fx = np.where(f_rgba[..., 3] >= 60)[1]
        fw = float(fx.max() - fx.min() + 1)
        ow = min(max(mw * 1.45, fw * 0.165), fw * 0.23)
    else:
        ow = max(mw * 1.50, 32.0)

    hw, ul, ld, cy0 = ow * 0.5, ow * 0.14, ow * 0.56, -ow * 0.02
    SS, pad = 4, int(math.ceil(ow * 1.15))
    bx0, by0 = max(0, int(round(cx - hw - pad))), max(0, int(round(cy - ul - pad)))
    bx1, by1 = min(W, int(round(cx + hw + pad))), min(H, int(round(cy + ld + pad)))
    lw, lh = bx1 - bx0, by1 - by0
    sw, sh = lw * SS, lh * SS
    cr, sr = math.cos(roll), math.sin(roll)
    to_s = lambda lx, ly: ((cx + lx * cr - ly * sr - bx0) * SS, (cy + lx * sr + ly * cr - by0) * SS)
    yys, xxs = np.mgrid[0:sh, 0:sw].astype(np.float32)
    dxs, dys = bx0 + xxs / SS - cx, by0 + yys / SS - cy
    uloc, vloc = dxs * cr + dys * sr, -dxs * sr + dys * cr

    ts = np.linspace(-1.0, 1.0, 81)
    yu = cy0 - ul * ((1 - ts ** 2) ** 0.75 * (1 - 0.10 * np.exp(-((ts / 0.22) ** 2))))
    yl = cy0 + ld * ((1 - ts ** 2) ** 0.65)
    upts = [to_s(x, y) for x, y in zip(ts * hw, yu)]
    lpts = [to_s(x, y) for x, y in zip(ts * hw, yl)]

    cm = Image.new("L", (sw, sh), 0)
    ImageDraw.Draw(cm).polygon(upts + lpts[::-1], fill=255)
    vn = np.clip((vloc - (cy0 - ul)) / max(1.0, ul + ld), 0, 1)[..., None]
    cav = np.zeros((sh, sw, 4), np.uint8)
    cav[..., :3] = np.clip(np.array([118, 46, 62]) * (1 - vn) + np.array([186, 78, 94]) * vn, 0, 255).astype(np.uint8)
    cav[..., 3] = np.array(cm, np.uint8)

    tim = Image.new("RGBA", (sw, sh), (0, 0, 0, 0))
    thw = hw * 0.88
    tts = np.linspace(-1, 1, 41)
    tb = (cy0 + ld * 0.02) + (ld * 0.09) * ((1 - tts ** 2) ** 0.7)
    ImageDraw.Draw(tim).polygon(
        [to_s(-thw, cy0 - ul * 1.45), to_s(thw, cy0 - ul * 1.45)] + [to_s(x, y) for x, y in zip((tts * thw)[::-1], tb[::-1])],
        fill=(252, 249, 248, 255),
    )
    ImageDraw.Draw(tim).line([to_s(x, y) for x, y in zip(tts * thw, tb)], fill=(218, 198, 204, 210), width=max(2, int(round(ow * 0.018 * SS))))

    # 圆润 U 形二次元舌尖 + 柔和玫瑰棕舌尖描边（确保 ParamTongueOut=1.0 吐舌外伸时圆润自然）
    gim = Image.new("RGBA", (sw, sh), (0, 0, 0, 0))
    ghw = hw * 0.78
    gts = np.linspace(-1.0, 1.0, 61)
    gy_top = (cy0 + ld * 0.78) - (ld * 0.46) * ((1 - gts ** 2) ** 0.55 * (1 - 0.06 * np.exp(-((gts / 0.18) ** 2))))
    gy_bot = (cy0 + ld * 0.78) + (ld * 0.56) * ((1 - gts ** 2) ** 0.62)
    ImageDraw.Draw(gim).polygon(
        [to_s(x, y) for x, y in zip(gts * ghw, gy_top)] + [to_s(x, y) for x, y in zip((gts * ghw)[::-1], gy_bot[::-1])],
        fill=(240, 138, 152, 255),
    )
    garr = np.array(gim, np.float32)
    hi = np.exp(-(((uloc / max(1.0, ghw * 0.65)) ** 2 + ((vloc - (cy0 + ld * 0.58)) / max(1.0, ld * 0.34)) ** 2) * 1.4))[..., None]
    garr[..., :3] = garr[..., :3] * (1 - 0.35 * hi) + np.array([255, 204, 210]) * (0.35 * hi)
    gim = Image.fromarray(np.clip(garr, 0, 255).astype(np.uint8))
    ImageDraw.Draw(gim).line([to_s(x, y) for x, y in zip(gts * ghw, gy_bot)], fill=(188, 92, 108, 215), width=max(2, int(round(ow * 0.018 * SS))))

    def mk_line(pts, bw, fade=False):
        im = Image.new("RGBA", (sw, sh), (0, 0, 0, 0))
        dr = ImageDraw.Draw(im)
        for i in range(len(pts) - 1):
            tm = -1.0 + 2.0 * (i + 0.5) / (len(pts) - 1)
            r = max(1.0, bw * (0.70 + 0.35 * (1 - tm ** 2)) * 0.5)
            c = (lcol[0], lcol[1], lcol[2], int(round(255 * (0.65 if (fade and abs(tm) < 0.38) else 1.0))))
            dr.line([pts[i], pts[i + 1]], fill=c, width=max(2, int(round(r * 2))))
            dr.ellipse((pts[i][0] - r, pts[i][1] - r, pts[i][0] + r, pts[i][1] + r), fill=c)
        return im

    ul_im = mk_line(upts, max(2, int(round(ow * 0.034 * SS))), False)
    ll_im = mk_line(lpts, max(2, int(round(ow * 0.022 * SS))), True)

    def put(im_s):
        full = np.zeros((H, W, 4), np.uint8)
        full[by0:by1, bx0:bx1] = np.array(im_s.resize((lw, lh), Image.Resampling.LANCZOS))
        return full

    print(f"✅ [自动口腔补全] 闭口中心=({cx:.0f},{cy:.0f}), 张口宽={ow:.1f}px, 头倾角={math.degrees(roll):.1f}°（纯 ASCII 命名 + 圆润 U 形舌）")
    return {
        "mouth": put(Image.fromarray(cav)),
        "tongue": put(gim),
        "upper teeth": put(tim),
        "lower lip line": put(ll_im),
        "upper lip line": put(ul_im),
    }

synth_mouth = synth_anime_mouth_ascii(raw_lyrs["mouth"], raw_lyrs.get("face"), raw_lyrs, axis_x) if (AUTO_MOUTH and "mouth" in raw_lyrs) else None

psd_layers = []
for t in ANATOMY:
    if t not in raw_lyrs:
        continue
    if t == "mouth" and synth_mouth is not None:
        for mn in ("mouth", "tongue", "upper teeth", "lower lip line", "upper lip line"):
            tr = trim_rgba(synth_mouth[mn])
            if tr:
                psd_layers.append((mn, *tr))
        trc = trim_rgba(raw_lyrs[t])
        if trc:
            psd_layers.append(("mouth close", *trc))
        continue
    if t in BILATERAL:
        sp = split_lr(raw_lyrs[t], axis_x)
        if sp:
            for suf, sub in (("-r", sp[0]), ("-l", sp[1])):
                tr = trim_rgba(sub)
                if tr:
                    psd_layers.append((f"{t}{suf}", *tr))
            continue
    tr = trim_rgba(raw_lyrs[t])
    if tr:
        psd_layers.append((t, *tr))

ready_psd = R1_DIR / f"{MODEL_NAME}_psd2live_ready.psd"
psd = PSDImage.new(mode="RGBA", size=(cw, ch), depth=8)
for nm, arr_c, lx, ty in psd_layers:
    safe_nm = nm.encode("ascii", "ignore").decode("ascii").strip() or "layer"
    psd.create_pixel_layer(Image.fromarray(arr_c), name=safe_nm, top=int(ty), left=int(lx), opacity=255)
psd.save(str(ready_psd))

chk = PSDImage.open(str(ready_psd))
chk_layers = list(chk)
print(f"✅ 已生成并回读校验完整 PSD: {ready_psd} ({ready_psd.stat().st_size / (1024*1024):.2f} MB, {len(chk_layers)} 层)")
assert ready_psd.stat().st_size > 100_000 and len(chk_layers) >= 10, "PSD 大小或图层数校验未通过！"

for sub in ("cubism_cmo3_moc3", "vtube_studio_pack", "web_player", "preview_animations", "mouth_check"):
    shutil.rmtree(R1_DIR / sub, ignore_errors=True)

subprocess.run([str(P2L_BIN), "--input", str(ready_psd), "--output", str(R1_DIR / "cubism_cmo3_moc3"), "--lang", "zh", "--atlas", "4096", "--mesh-spacing", "44"], check=True)
subprocess.run([str(P2L_BIN), "export", str(ready_psd), "--target", "vtube-studio", "--output", str(R1_DIR / "vtube_studio_pack"), "--name", MODEL_NAME], check=False)
subprocess.run([str(P2L_BIN), "export", str(ready_psd), "--target", "web", "--output", str(R1_DIR / "web_player"), "--name", MODEL_NAME], check=False)

r1_anim = R1_DIR / "preview_animations"
r1_anim.mkdir(exist_ok=True)
for clip in ("Nod", "Shake"):
    subprocess.run([
        str(P2L_BIN), "export", str(ready_psd), "--target", "mp4",
        "--output", str(r1_anim), "--name", f"{MODEL_NAME}_{clip.lower()}",
        "--set", f"clip={clip}", "--set", "fps=30", "--set", "size=768",
        "--set", "physics=true", "--set", "background=ffffffff",
    ], check=False)

m_dir = R1_DIR / "mouth_check"
m_dir.mkdir(exist_ok=True)
m_imgs = []
for mid, expr, lbl in [
    ("m0", "ParamMouthOpenY=0,ParamMouthForm=0", "Closed (Orig)"),
    ("m1", "ParamMouthOpenY=0.55,ParamMouthForm=0.8", "Smile Open"),
    ("m2", "ParamMouthOpenY=1.0,ParamMouthForm=0.3", "Wide (Teeth+Tongue)"),
    ("m3", "ParamMouthOpenY=0.9,ParamMouthForm=0.5,ParamTongueOut=1.0", "Tongue Out"),
]:
    subprocess.run([str(P2L_BIN), "export", str(ready_psd), "--target", "psd-pose", "--output", str(m_dir), "--name", mid, "--set", f"pose={expr}", "--set", "scale=0.6"], check=False)
    pf = m_dir / f"{mid}.psd"
    if pf.exists():
        c_im = PSDImage.open(str(pf)).composite().convert("RGBA")
        bg = Image.new("RGBA", c_im.size, (248, 248, 250, 255))
        bg.alpha_composite(c_im)
        rw, rh = bg.size
        m_imgs.append((lbl, bg.crop((int(rw * 0.28), int(rh * 0.10), int(rw * 0.72), int(rh * 0.54))).convert("RGB")))
if m_imgs:
    fig, axes = plt.subplots(1, len(m_imgs), figsize=(4 * len(m_imgs), 4.3), dpi=120)
    for ax, (lbl, im) in zip(np.atleast_1d(axes), m_imgs):
        ax.imshow(im); ax.set_title(lbl); ax.axis("off")
    plt.tight_layout()
    fig.savefig(r1_anim / "mouth_4states.png", bbox_inches="tight")
    plt.close(fig)
shutil.rmtree(m_dir, ignore_errors=True)
""".lstrip(), encoding="utf-8")

        run_stream([
            sys.executable, "-u", str(P2L_SCRIPT),
            "--r1-dir", str(R1_DIR), "--p2l-bin", str(P2L_BIN),
            "--model-name", MODEL_NAME, "--mode", ROUTE1_IMAGE_MODE,
            "--auto-mouth", "1" if AUTO_SYNTH_MOUTH else "0",
        ])

        r1_anim = R1_DIR / "preview_animations"
        show_img_b64(ST_OUT / "reconstruction.png", "【路线 1】See-through 23 层拆分重拼图", max_w=440)
        show_img_b64(r1_anim / "mouth_4states.png", "【路线 1】闭口自动补全口腔/上牙/圆润 U 形舌 4 态验收", max_w=920)
        for clip in ("nod", "shake"):
            show_video_b64(r1_anim / f"{MODEL_NAME}_{clip}.mp4", f"【路线 1】真 Live2D 物理动画 ({clip.upper()})", width=440)
        STATUS["Route 1 (See-through + PSD2Live)"] = "✅ 成功"
    except Exception as e:
        STATUS["Route 1 (See-through + PSD2Live)"] = f"❌ 失败: {e}"
        traceback.print_exc()

# ==============================================================================
# 路线 2：Talking Head Anime 4（独立子进程 + 无损前景红框对齐 + THA3 兜底）
# ==============================================================================
if RUN_ROUTE_2:
    print("\n" + "#" * 74)
    print("### [路线 2] Talking Head Anime 4 (二次元 45 参数神经人偶)")
    print("#" * 74)
    try:
        R2_DIR = OUT_ALL / "02_talking_head_anime"
        R2_DIR.mkdir(parents=True, exist_ok=True)
        R2_SCRIPT = SCRIPTS_DIR / "run_route2_tha4.py"
        R2_SCRIPT.write_text(r"""
import math, shutil, subprocess, sys, zipfile
from pathlib import Path
import cv2
import numpy as np
import torch
from PIL import Image, ImageDraw
import imageio.v2 as imageio

WORK = Path("/content/ai_live2d_3routes")
R1_ST = WORK / "outputs" / "01_seethrough_psd2live" / "seethrough_layers"
R2_DIR = WORK / "outputs" / "02_talking_head_anime"
SRC_FULL = WORK / "assets" / "source_full.png"
THA4_REPO = Path("/content/talking-head-anime-4-demo")

if not THA4_REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pkhungurn/talking-head-anime-4-demo.git", str(THA4_REPO)], check=True)

wdir4 = THA4_REPO / "data" / "tha4"
wdir4.mkdir(parents=True, exist_ok=True)
wfiles4 = ["eyebrow_decomposer.pt","eyebrow_morphing_combiner.pt","face_morpher.pt","body_morpher.pt","upscaler.pt"]

if not all((wdir4 / f).exists() for f in wfiles4):
    z4 = WORK / "tha4-models.zip"
    url4 = "https://www.dropbox.com/scl/fi/7wec0sur7449iqgtlpi3n/tha4-models.zip?rlkey=0f9d1djmbvjjjn09469s1adx8&dl=1"
    try:
        if not (z4.exists() and zipfile.is_zipfile(z4) and z4.stat().st_size > 500_000_000):
            subprocess.run(["curl", "-L", "--retry", "3", "--fail", "-o", str(z4), url4], check=True)
        if zipfile.is_zipfile(z4) and z4.stat().st_size > 500_000_000:
            tmp4 = WORK / "_tha4_tmp"
            with zipfile.ZipFile(z4) as zf:
                zf.extractall(tmp4)
            for fn in wfiles4:
                shutil.move(str(next(tmp4.rglob(fn))), str(wdir4 / fn))
            shutil.rmtree(tmp4, ignore_errors=True)
            z4.unlink(missing_ok=True)
    except Exception as exc:
        print(f"⚠️ THA4 Dropbox 下载受限 ({exc})，自动切至 HuggingFace THA3 镜像！", flush=True)

use_tha4 = all((wdir4 / f).exists() for f in wfiles4)
def safe_tload(fn):
    with open(fn, "rb") as f:
        return torch.load(f, map_location="cpu", weights_only=False)

if use_tha4:
    sys.path.insert(0, str(THA4_REPO / "src"))
    import tha4.shion.core.load_save as t4_ls
    import tha4.poser.modes.mode_07 as m07
    from tha4.poser.modes.pose_parameters import get_pose_parameters as get_pp
    from tha4.shion.base.image_util import (
        extract_pytorch_image_from_PIL_image as ext_t4,
        convert_pytorch_image_to_zero_to_one_numpy_image as t4_01,
        convert_zero_to_one_numpy_image_to_PIL_image as t4_pil,
    )
    t4_ls.torch_load = safe_tload; m07.torch_load = safe_tload
    poser = m07.create_poser(device=torch.device("cuda"), module_file_names={k.replace(".pt",""): str(wdir4 / k) for k in wfiles4})
    poser.get_modules(); pparams = get_pp(); eng_tag = "THA4"
else:
    from huggingface_hub import hf_hub_download
    THA3_REPO = Path("/content/talking-head-anime-3-demo")
    if not THA3_REPO.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pkhungurn/talking-head-anime-3-demo.git", str(THA3_REPO)], check=True)
    wdir3 = THA3_REPO / "data" / "models" / "standard_float"; wdir3.mkdir(parents=True, exist_ok=True)
    wfiles3 = ["eyebrow_decomposer.pt","eyebrow_morphing_combiner.pt","face_morpher.pt","two_algo_face_body_rotator.pt","editor.pt"]
    for fn in wfiles3:
        if not (wdir3 / fn).exists():
            shutil.copy2(hf_hub_download("OktayAlpk/talking-head-anime-3", f"standard_float/{fn}"), wdir3 / fn)
    sys.path.insert(0, str(THA3_REPO))
    import tha3.util as t3u, tha3.poser.modes.standard_float as sf3
    from tha3.poser.modes.pose_parameters import get_pose_parameters as get_pp
    t3u.torch_load = safe_tload; sf3.torch_load = safe_tload
    poser = sf3.create_poser(device=torch.device("cuda"), module_file_names={k.replace(".pt",""): str(wdir3 / k) for k in wfiles3})
    poser.get_modules(); pparams = get_pp(); eng_tag = "THA3"

pidx = {}
ki = 0
for g in pparams.get_pose_parameter_groups():
    for nm in g.get_parameter_names():
        pidx[nm] = ki; ki += 1

# 优先复用路线 1 无损原图前景 + face.png 精准红框对齐；若未开路线 1 则走 CPU rembg 兜底
if (R1_ST / "src_img.png").exists() and (R1_ST / "face.png").exists():
    src_rgb = np.array(Image.open(R1_ST / "src_img.png").convert("RGB"))
    H, W = src_rgb.shape[:2]
    fg_alpha = np.zeros((H, W), np.uint8)
    for fn in ["back hair","bottomwear","topwear","handwear","neck","ears","earwear","face","mouth","eyewhite","irides","eyelash","eyebrow","front hair","headwear"]:
        p = R1_ST / f"{fn}.png"
        if p.exists():
            a = np.array(Image.open(p).convert("RGBA"))[..., 3]
            a[a < 55] = 0
            fg_alpha = np.maximum(fg_alpha, a)
    m_fg = (fg_alpha >= 55).astype(np.uint8)
    nl, lbl, st, _ = cv2.connectedComponentsWithStats(m_fg, connectivity=8)
    clean_fg = np.zeros_like(m_fg, dtype=bool)
    for k in range(1, nl):
        if st[k, cv2.CC_STAT_AREA] >= 400:
            clean_fg |= (lbl == k)
    fg_alpha[~clean_fg] = 0
    rgba_src = Image.fromarray(np.dstack([src_rgb, fg_alpha]), "RGBA")

    f_arr = np.array(Image.open(R1_ST / "face.png").convert("RGBA"))[..., 3]
    fm = (f_arr >= 60).astype(np.uint8)
    nl2, lbl2, st2, _ = cv2.connectedComponentsWithStats(fm, connectivity=8)
    best_k = 1 + int(np.argmax(st2[1:, cv2.CC_STAT_AREA]))
    fy, fx = np.where(lbl2 == best_k)
    fx0, fx1, fy0, fy1 = float(fx.min()), float(fx.max()), float(fy.min()), float(fy.max())
    fcx, fh = (fx0 + fx1) * 0.5, max(10.0, fy1 - fy0)
    scale = 96.0 / fh
    scaled = rgba_src.resize((max(1, int(round(W * scale))), max(1, int(round(H * scale)))), Image.Resampling.LANCZOS)
    in512 = Image.new("RGBA", (512, 512), (0, 0, 0, 0))
    in512.paste(scaled, (int(round(256.0 - fcx * scale)), int(round(192.0 - fy1 * scale))))
else:
    from rembg import remove, new_session
    rgba_cut = remove(Image.open(SRC_FULL).convert("RGBA"), session=new_session("isnet-anime", providers=["CPUExecutionProvider"]))
    w0, h0 = rgba_cut.size
    csz = int(round(min(w0, h0) * 0.97))
    x0 = int(round(w0 / 2.0 + w0 * 0.008 - csz / 2.0))
    y0 = int(round(-0.02 * csz))
    c512 = Image.new("RGBA", (csz, csz), (0, 0, 0, 0)); c512.paste(rgba_cut, (-x0, -y0))
    in512 = c512.resize((512, 512), Image.Resampling.LANCZOS)

a512 = np.array(in512, np.uint8); a512[a512[..., 3] <= 2] = 0
in512 = Image.fromarray(a512, "RGBA")
in512.save(R2_DIR / "input_512_rgba.png")

guide = Image.alpha_composite(Image.new("RGBA", (512, 512), (245, 246, 250, 255)), in512).convert("RGB")
gd = ImageDraw.Draw(guide)
gd.rectangle([160, 32, 352, 224], outline=(0, 175, 70), width=2)
gd.rectangle([192, 64, 320, 192], outline=(235, 35, 35), width=2)
guide.save(R2_DIR / "alignment_guide.png")

if use_tha4:
    in_ten = ext_t4(in512.copy(), premultiply_alpha=True, perform_srgb_to_linear=True).to("cuda", dtype=poser.get_dtype())
else:
    in_ten = t3u.extract_pytorch_image_from_PIL_image(in512.copy()).to("cuda", dtype=poser.get_dtype())

def render_tha(pdict):
    vec = np.zeros(45, np.float32)
    for k, v in pdict.items():
        vec[pidx[k]] = float(v)
    pt = torch.from_numpy(vec).to("cuda", dtype=poser.get_dtype()).unsqueeze(0)
    with torch.no_grad():
        out = poser.pose(in_ten, pt)[0].detach().float().cpu()
    if use_tha4:
        rgba = t4_pil(t4_01(out), use_straight_alpha=True, perform_linear_to_srgb=True)
    else:
        rgba = Image.fromarray(np.uint8(np.clip(np.rint(t3u.rgba_to_numpy_image(out) * 255.0), 0, 255)), "RGBA")
    bg = Image.new("RGBA", (512, 512), (238, 241, 248, 255))
    return Image.alpha_composite(bg, rgba).convert("RGB")

specs = [
    ("Rest", {}), ("Blink", {"eye_wink_left":1,"eye_wink_right":1}),
    ("Wink", {"eye_happy_wink_right":1,"eyebrow_happy_left":0.65,"eyebrow_happy_right":0.65,"mouth_raised_corner_left":0.85,"mouth_raised_corner_right":0.85,"neck_z":-0.35}),
    ("Happy", {"eye_happy_wink_left":0.95,"eye_happy_wink_right":0.95,"eyebrow_happy_left":0.85,"eyebrow_happy_right":0.85,"mouth_aaa":0.75,"mouth_raised_corner_left":0.95,"mouth_raised_corner_right":0.95}),
    ("Surprised", {"eye_surprised_left":1,"eye_surprised_right":1,"eyebrow_raised_left":1,"eyebrow_raised_right":1,"iris_small_left":0.45,"iris_small_right":0.45,"mouth_ooo":0.9}),
    ("Angry", {"eyebrow_angry_left":1,"eyebrow_angry_right":1,"eye_unimpressed_left":0.65,"eye_unimpressed_right":0.65,"mouth_lowered_corner_left":0.85,"mouth_lowered_corner_right":0.85,"head_y":0.4,"iris_rotation_y":-0.65}),
    ("Speak A/O", {"mouth_aaa":0.85,"mouth_ooo":0.35,"eyebrow_raised_left":0.35,"eyebrow_raised_right":0.35}),
    ("Turn Left", {"head_y":0.8,"body_y":0.45,"iris_rotation_y":-0.75,"mouth_raised_corner_left":0.5,"mouth_raised_corner_right":0.5}),
    ("Turn Right+Tilt", {"head_y":-0.8,"body_y":-0.4,"neck_z":-0.55,"iris_rotation_y":0.55}),
]
grid = Image.new("RGB", (512 * 3, 512 * 3), (24, 28, 38))
for i, (lb, pd) in enumerate(specs):
    tile = render_tha(pd)
    ImageDraw.Draw(tile).rectangle([0, 0, 230, 32], fill=(20, 24, 34))
    ImageDraw.Draw(tile).text((10, 8), f"[{eng_tag}] {lb}", fill=(255, 255, 255))
    r, c = divmod(i, 3)
    grid.paste(tile, (c * 512, r * 512))
grid.save(R2_DIR / "tha_3x3_expressions.png")

pulse = lambda u: 0.5 * (1.0 - math.cos(2.0 * math.pi * max(0.0, min(1.0, u))))
seq = []
for i in range(60):
    t = i / 30.0
    bl = pulse((t - 0.5) / 0.28) if 0.5 <= t <= 0.78 else (pulse((t - 1.3) / 0.28) if 1.3 <= t <= 1.58 else 0.0)
    seq.append({"breathing": pulse(t / 2.0), "eye_wink_left": bl, "eye_wink_right": bl, "iris_rotation_y": 0.45 * math.sin(math.pi * t)})
for vk in ("mouth_aaa", "mouth_iii", "mouth_uuu", "mouth_eee", "mouth_ooo"):
    for i in range(15):
        w = pulse(i / 14.0); seq.append({vk: 0.95 * w, "breathing": 0.3 + 0.3 * w})
for i in range(75):
    u = i / 74.0; hy = 0.85 * math.sin(2 * math.pi * u)
    seq.append({"head_y": hy, "head_x": 0.45 * math.sin(4 * math.pi * u), "neck_z": 0.45 * math.sin(2 * math.pi * u + 0.7),
                "iris_rotation_y": float(np.clip(-0.65 * hy, -1, 1)), "mouth_raised_corner_left": 0.4, "mouth_raised_corner_right": 0.4})
for i in range(54):
    u = i / 53.0; env = math.sin(math.pi * u)
    seq.append({"body_y": 0.75 * math.sin(2 * math.pi * u) * env, "body_z": 0.55 * math.sin(2 * math.pi * u) * env, "breathing": pulse(u)})

mp4_out = R2_DIR / "tha_live2d_30fps.mp4"
with imageio.get_writer(str(mp4_out), fps=30, codec="libx264", pixelformat="yuv420p", quality=8, macro_block_size=1) as wr:
    for pd in seq:
        wr.append_data(np.asarray(render_tha(pd)))
print(f"✅ 路线 2 ({eng_tag}) 完成！输出: {mp4_out}", flush=True)
""".lstrip(), encoding="utf-8")

        run_stream([sys.executable, "-u", str(R2_SCRIPT)])
        show_img_b64(R2_DIR / "alignment_guide.png", "【路线 2】THA4 红框精准对齐检查", max_w=340)
        show_img_b64(R2_DIR / "tha_3x3_expressions.png", "【路线 2】THA4 3×3 核心表情矩阵", max_w=760)
        show_video_b64(R2_DIR / "tha_live2d_30fps.mp4", "【路线 2】THA4 30fps 五科目动态评测", width=450)
        STATUS["Route 2 (Talking Head Anime)"] = "✅ 成功"
    except Exception as e:
        STATUS["Route 2 (Talking Head Anime)"] = f"❌ 失败: {e}"
        traceback.print_exc()

# ==============================================================================
# 路线 3：LivePortrait（独立子进程 + CPU ONNX 安全检脸 + GPU PyTorch 极速渲染）
# ==============================================================================
if RUN_ROUTE_3:
    print("\n" + "#" * 74)
    print("### [路线 3] KlingAIResearch/LivePortrait (驱动重演 + 零视频滑块)")
    print("#" * 74)
    try:
        R3_DIR = OUT_ALL / "03_liveportrait"
        R3_DIR.mkdir(parents=True, exist_ok=True)
        R3_SCRIPT = SCRIPTS_DIR / "run_route3_liveportrait.py"
        R3_SCRIPT.write_text(r"""
import math, os, subprocess, sys
from pathlib import Path
import torch
import onnxruntime as ort

_orig_sess = ort.InferenceSession
_avail = set(ort.get_available_providers())
class _SafeSession(_orig_sess):
    def __init__(self, path_or_bytes, sess_options=None, providers=None, provider_options=None, **kw):
        if providers is not None:
            filt = [p for p in providers if (p[0] if isinstance(p, tuple) else p) in _avail]
            if filt:
                providers = filt
            else:
                providers = ["CPUExecutionProvider"]
                provider_options = None
        super().__init__(path_or_bytes, sess_options=sess_options, providers=providers, provider_options=provider_options, **kw)
ort.InferenceSession = _SafeSession

from huggingface_hub import snapshot_download

WORK = Path("/content/ai_live2d_3routes")
SRC_BUST = WORK / "assets" / "source_bust_1024.png"
R3_DIR = WORK / "outputs" / "03_liveportrait"
LP_REPO = Path("/content/LivePortrait")

if not (LP_REPO / "inference.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/KlingAIResearch/LivePortrait.git", str(LP_REPO)], check=True)

wdir = LP_REPO / "pretrained_weights"
if not (wdir / "liveportrait/base_models/spade_generator.pth").exists():
    for hfr in ("KlingTeam/LivePortrait", "KwaiVGI/LivePortrait"):
        try:
            snapshot_download(repo_id=hfr, local_dir=str(wdir), ignore_patterns=["*.git*", "README.md", "docs/*", "liveportrait_animals/*"])
            break
        except Exception as e:
            print(f"⚠️ {hfr} 下载失败: {e}", flush=True)

os.chdir(LP_REPO)
sys.path.insert(0, str(LP_REPO))

from src.config.argument_config import ArgumentConfig
from src.config.inference_config import InferenceConfig
from src.config.crop_config import CropConfig
from src.live_portrait_pipeline import LivePortraitPipeline
from src.gradio_pipeline import GradioPipeline
from src.utils.camera import get_rotation_matrix
from src.utils.crop import paste_back
from src.utils.video import images2video

inf_cfg = InferenceConfig(flag_use_half_precision=True, flag_force_cpu=False, flag_stitching=True, flag_pasteback=True, flag_do_crop=True, driving_option="expression-friendly", driving_multiplier=0.95)
crop_cfg = CropConfig(det_thresh=0.05, scale=2.5, vy_ratio=-0.125, flag_force_cpu=True)
lp_pipe = LivePortraitPipeline(inference_cfg=inf_cfg, crop_cfg=crop_cfg)
lp_pipe.cropper.face_analysis_wrapper.det_model.det_thresh = 0.05

for drv in ("assets/examples/driving/d0.mp4", "assets/examples/driving/d11.mp4"):
    a = ArgumentConfig(source=str(SRC_BUST), driving=drv, output_dir=str(R3_DIR), det_thresh=0.05, scale=2.5, vy_ratio=-0.125, driving_option="expression-friendly", driving_multiplier=0.95)
    lp_pipe.execute(a)
del lp_pipe; torch.cuda.empty_cache()

args_lp = ArgumentConfig(det_thresh=0.05, scale=2.5, vy_ratio=-0.125, flag_use_half_precision=True, flag_force_cpu=False)
pipe_lp = GradioPipeline(inference_cfg=inf_cfg, crop_cfg=crop_cfg, args=args_lp)
pipe_lp.cropper.face_analysis_wrapper.det_model.det_thresh = 0.05
src_eye_r, src_lip_r = pipe_lp.init_retargeting_image(2.5, 0.0, 0.0, str(SRC_BUST))
f_s, x_s, R_s, _, xs_info, src_lmk, M_c2o, m_ori, img_ori = pipe_lp.prepare_retargeting_image(str(SRC_BUST), 0.0, 0.0, 0.0, 2.5, flag_do_crop=True)
dev = pipe_lp.live_portrait_wrapper.device
f_s, x_s, R_s = f_s.to(dev), x_s.to(dev), R_s.to(dev)
x_c_s, exp_b, sc_b = xs_info["kp"].to(dev), xs_info["exp"].to(dev), xs_info["scale"].to(dev)
t_b = xs_info["t"].clone().to(dev); t_b[..., 2] = 0.0
p_b, y_b, r_b = xs_info["pitch"].to(dev), xs_info["yaw"].to(dev), xs_info["roll"].to(dev)

@torch.no_grad()
def lp_frame(eye_r=None, lip_r=None, pitch=0.0, yaw=0.0, roll=0.0, smile=0.0, wink=0.0, eb_x=0.0, eb_y=0.0):
    dn = exp_b.clone()
    Rd = (get_rotation_matrix(p_b + pitch, y_b + yaw, r_b + roll).to(dev) @ R_s.permute(0, 2, 1)) @ R_s
    if eb_x or eb_y:
        dn = pipe_lp.update_delta_new_eyeball_direction(torch.tensor(float(eb_x), device=dev), torch.tensor(float(eb_y), device=dev), dn)
    if smile:
        dn = pipe_lp.update_delta_new_smile(torch.tensor(float(smile), device=dev), dn)
    if wink:
        dn = pipe_lp.update_delta_new_wink(torch.tensor(float(wink), device=dev), dn)
    xd = sc_b * (x_c_s @ Rd + dn) + t_b
    if eye_r is not None:
        xd = xd + pipe_lp.live_portrait_wrapper.retarget_eye(x_s, pipe_lp.live_portrait_wrapper.calc_combined_eye_ratio([[float(eye_r)]], src_lmk))
    if lip_r is not None:
        xd = xd + pipe_lp.live_portrait_wrapper.retarget_lip(x_s, pipe_lp.live_portrait_wrapper.calc_combined_lip_ratio([[float(lip_r)]], src_lmk))
    xd = pipe_lp.live_portrait_wrapper.stitching(x_s, xd)
    o512 = pipe_lp.live_portrait_wrapper.parse_output(pipe_lp.live_portrait_wrapper.warp_decode(f_s, x_s, xd)["out"])[0]
    return paste_back(o512, M_c2o, img_ori, m_ori)

bell = lambda u: 0.5 * (1.0 - math.cos(2.0 * math.pi * u))
frames = []
for i in range(30): frames.append(lp_frame(eye_r=max(0.02, float(src_eye_r) * (1 - 0.92 * bell(i / 29.0)))))
for i in range(30): frames.append(lp_frame(wink=24.0 * bell(i / 29.0), smile=0.7 * bell(i / 29.0)))
for i in range(35): frames.append(lp_frame(lip_r=min(0.75, float(src_lip_r) + 0.50 * bell(i / 34.0))))
for i in range(35): frames.append(lp_frame(eb_x=18.0 * math.sin(2 * math.pi * i / 34.0)))
for i in range(50): frames.append(lp_frame(yaw=20.0 * math.sin(2 * math.pi * i / 49.0), pitch=8.0 * math.sin(4 * math.pi * i / 49.0)))
images2video(frames, wfp=str(R3_DIR / "liveportrait_sliders.mp4"), fps=25)
print("✅ 路线 3 (LivePortrait) 全部完成！", flush=True)
""".lstrip(), encoding="utf-8")

        run_stream([sys.executable, "-u", str(R3_SCRIPT)])
        for drv_stem, tag in [("d0", "d0微笑眼神"), ("d11", "d11说话眨眼")]:
            show_video_b64(R3_DIR / f"{SRC_BUST.stem}--{drv_stem}_concat.mp4", f"【路线 3】LivePortrait 驱动重演 ({tag} 三联对比)", width=740)
        show_video_b64(R3_DIR / "liveportrait_sliders.mp4", "【路线 3】LivePortrait 零视频滑块扫参测试", width=450)
        STATUS["Route 3 (LivePortrait)"] = "✅ 成功"
    except Exception as e:
        STATUS["Route 3 (LivePortrait)"] = f"❌ 失败: {e}"
        traceback.print_exc()

# 汇总与打包下载
print("\n" + "=" * 74)
print("🏁 运行状态汇总：")
for k, v in STATUS.items():
    print(f"  • {k:<36s} : {v}")
print("=" * 74)

ZIP_PATH = WORK / f"{MODEL_NAME}_AI_Live2D_3Routes_Complete.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for p in sorted(OUT_ALL.rglob("*")):
        if p.is_file() and not p.stem.endswith("_web"):
            zf.write(p, p.relative_to(OUT_ALL).as_posix())
print(f"📦 完整产物包已生成: {ZIP_PATH} ({ZIP_PATH.stat().st_size / (1024*1024):.1f} MB)")

if DRIVE_OUT_DIR is not None:
    shutil.copy2(ZIP_PATH, DRIVE_OUT_DIR / ZIP_PATH.name)
    print(f"☁️ 已同步备份至 Google Drive: {DRIVE_OUT_DIR / ZIP_PATH.name}")

try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
except Exception:
    print("提示：请直接在左侧 Colab 文件栏右键下载上述 ZIP 文件。")
